# Switzerland Country Dashboard — Monthly Automation

End-to-end pipeline, driven by one raw monthly file:

1. **`raw_df`** — read the SANA monthly report CSV.
2. **Data Extract** — de-select the specified Column1/Column2 values *in the Data sheet* (rows hidden via AutoFilter), clean the numeric columns (`'` and `.00` removed), and build a refreshed **Pivot**. Loaded back as **`dataextract_df`**.
3. **`SwitzerlandMapping_df`** — read the mapping (dimension) file.
4. **Linked (Option B — resolve to values)** — join the Pivot rows with the mapping to produce the linked **Data** sheet as real values → **`CountryDashboardData_Switzerland_Linked_df`**, saved to disk.

**Dynamic window:** the Data sheet always shows the trailing **24 months** ending at the latest month in the raw file (override with `END_MONTH`). For the current file that is **Jul 2024 – Jun 2026**; next month's file shifts it to **Aug 2024 – Jul 2026**, and so on.

In [1]:
import re
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font

## Parameters
Edit these to point at a new month's raw file or to override the window.

In [2]:
UPLOADS = "/mnt/user-data/uploads"
OUTPUTS = "/mnt/user-data/outputs"

RAW_CSV      = f"{UPLOADS}/SANA_Monthly_Report__APO_SDA__Tabelle_1.csv"
MAPPING_XLSX = f"{UPLOADS}/Switzerland_Mapping.xlsx"
# Prior published dashboard -> source of the older 12 months of (frozen) history:
EXISTING_DASHBOARD = f"{UPLOADS}/Country_Dashboard_Data_-_Switzerland_June_2026.xlsx"

EXTRACT_XLSX = f"{OUTPUTS}/Switzerland_Data_Extract_Automated.xlsx"
LINKED_XLSX  = f"{OUTPUTS}/Country_Dashboard_Data_-_Switzerland_Linked_Automated.xlsx"
FINAL_XLSX   = f"{OUTPUTS}/Country_Dashboard_-_Switzerland_FINAL.xlsx"

WINDOW_MONTHS  = 24         # months coming from the Linked file
HISTORY_MONTHS = 12         # older months pulled from the existing dashboard
TOTAL_MONTHS   = 36         # final dashboard horizon
END_MONTH      = None        # None -> latest month in the raw file; else e.g. "Jun 2026"

# Values to DE-SELECT (excluded from the Pivot / hidden in the Data sheet)
COL1_DESELECT = {"SCHWEIZ", "SWITZERLAND_CENTRAL", "SWITZERLAND_EAST", "SWITZERLAND_WEST_TI"}
COL2_DESELECT = {"GT Santen", "GT Others", "GT Santen DED Market",
                 "S01E2 Myotica & Mittel zur Glaukombehandlung"}

# German -> English month abbreviations (raw file uses German)
DE = {"Jan":"Jan","Feb":"Feb","Mrz":"Mar","Mär":"Mar","Apr":"Apr","Mai":"May","Jun":"Jun",
      "Jul":"Jul","Aug":"Aug","Sep":"Sep","Okt":"Oct","Nov":"Nov","Dez":"Dec"}

def en_month(lbl):
    m, y = lbl.split(); yr = int(y); yr += 2000 if yr < 100 else 0
    return f"{DE[m]} {yr}"

## Step 0 — `raw_df`
The CSV is latin-1 encoded and `;`-delimited. Row 1 = category (Umsatz/Menge), row 2 = month, rows 3+ = data (region × product). Kept 1:1 with the Data-sheet layout: headers `Column1…ColumnN`, with the category and month rows as the first two body rows.

In [3]:
_raw = pd.read_csv(RAW_CSV, sep=";", encoding="latin-1", header=None,
                   dtype=str, keep_default_na=False)
n_cols = _raw.shape[1]
col_names = [f"Column{i+1}" for i in range(n_cols)]
raw_df = _raw.copy(); raw_df.columns = col_names

category_row = raw_df.iloc[0].tolist()
month_row    = raw_df.iloc[1].tolist()
data_df      = raw_df.iloc[2:].reset_index(drop=True).copy()

label_cols = ["Column1", "Column2"]
value_cols = [c for c in col_names if c not in label_cols]
print("raw_df:", raw_df.shape)
raw_df.head()

raw_df: (10628, 74)


,Column1,Column2,Column3,Column4,Column5,Column6,Column7,Column8,Column9,Column10,...,Column65,Column66,Column67,Column68,Column69,Column70,Column71,Column72,Column73,Column74
0,,,Umsatz,Umsatz,Umsatz,Umsatz,Umsatz,Umsatz,Umsatz,Umsatz,...,Menge,Menge,Menge,Menge,Menge,Menge,Menge,Menge,Menge,Menge
1,,,Jul 23,Aug 23,Sep 23,Okt 23,Nov 23,Dez 23,Jan 24,Feb 24,...,Sep 25,Okt 25,Nov 25,Dez 25,Jan 26,Feb 26,Mrz 26,Apr 26,Mai 26,Jun 26
2,SCHWEIZ,S01E2 Myotica & Mittel zur Glaukombehandlung,2'259'785.00,2'314'303.00,2'326'808.00,2'378'085.00,2'567'515.00,2'628'446.00,2'237'180.00,2'179'686.00,...,93'917.00,92'605.00,92'290.00,109'097.00,83'960.00,85'689.00,97'705.00,88'764.00,91'787.00,102'885.00
3,SCHWEIZ,GT Santen,632'302.00,617'885.00,635'889.00,643'276.00,715'558.00,715'500.00,598'155.00,563'290.00,...,28'635.00,27'587.00,27'657.00,31'719.00,24'493.00,24'581.00,28'985.00,26'832.00,27'167.00,29'823.00
4,SCHWEIZ,Cosopt,166'472.00,162'497.00,164'919.00,151'193.00,202'007.00,168'702.00,129'821.00,104'206.00,...,4'943.00,4'410.00,4'571.00,5'048.00,3'804.00,3'736.00,4'524.00,4'031.00,3'960.00,4'425.00


### Dynamic 24-month window
`Umsatz` (revenue) and `Menge` (units) each span the same months. We take the trailing `WINDOW_MONTHS` ending at `END_MONTH` (default = latest month present).

In [4]:
umsatz_idx = [i for i, c in enumerate(category_row) if c == "Umsatz"]  # 0-based positions
menge_idx  = [i for i, c in enumerate(category_row) if c == "Menge"]
month_labels = [month_row[i] for i in umsatz_idx]
n_months = len(umsatz_idx)

if END_MONTH is None:
    end_pos = n_months - 1
else:
    end_pos = [en_month(month_labels[i]) for i in range(n_months)].index(END_MONTH)
start_pos = end_pos - WINDOW_MONTHS + 1
assert start_pos >= 0, "Not enough months in the raw file for the requested window."

win        = list(range(start_pos, end_pos + 1))
win_labels = [en_month(month_labels[i]) for i in win]
units_src_cols = [col_names[menge_idx[i]]  for i in win]   # Menge  (units)
rev_src_cols   = [col_names[umsatz_idx[i]] for i in win]    # Umsatz (revenue)
print(f"Window: {win_labels[0]} -> {win_labels[-1]} ({WINDOW_MONTHS} months)")

Window: Jul 2024 -> Jun 2026 (24 months)


## Steps 1–3 — clean numeric columns (CTRL+H)
On every value column (all columns except `Column1`/`Column2`): remove `'` then `.00`, and convert to numbers (blank → NaN).

In [5]:
data_clean = data_df.copy()
for c in value_cols:
    data_clean[c] = (data_clean[c].astype(str)
                     .str.replace("'", "", regex=False)
                     .str.replace(".00", "", regex=False))

data_num = data_clean.copy()
for c in value_cols:
    data_num[c] = pd.to_numeric(data_num[c].replace("", np.nan), errors="coerce")
print("cleaned data rows:", len(data_clean))

cleaned data rows: 10626


## Steps 1–2 & 4 — Pivot (de-selection applied, refreshed)
Exclude the de-selected Column1/Column2 values and any blanks, then aggregate by `Column1`×`Column2`. Value columns follow the reference Pivot: **Units** (Menge, trailing 24) then **Revenue** (Umsatz, trailing 24).

In [6]:
mask = (~data_num.Column1.isin(COL1_DESELECT)
        & ~data_num.Column2.isin(COL2_DESELECT)
        & data_num.Column1.astype(str).str.strip().ne("")
        & data_num.Column2.astype(str).str.strip().ne(""))
pivot_src = data_num[mask].copy()

pivot_value_cols = units_src_cols + rev_src_cols
pivot = (pivot_src.groupby(["Column1", "Column2"], as_index=False)[pivot_value_cols].sum()
                  .sort_values(["Column1", "Column2"], kind="mergesort")
                  .reset_index(drop=True))
print(f"Pivot: {pivot.Column1.nunique()} bricks x {pivot.Column2.nunique()} products = {len(pivot)} rows")

Pivot: 227 bricks x 42 products = 9534 rows


## Write the Data Extract workbook
**Data** sheet: all rows, cleaned numbers, de-selected rows *hidden* via AutoFilter (matches the reference — rows remain physically present). **Pivot** sheet: the refreshed aggregation. Reloaded as **`dataextract_df`**.

In [7]:
wb = openpyxl.Workbook(); wsP = wb.active; wsP.title = "Pivot"
nU, nR = len(units_src_cols), len(rev_src_cols)
row2 = [None, None] + [f"Units - Month {i+1}" for i in range(nU)] + [f"Revenue - Month {i+1}" for i in range(nR)]
for j, v in enumerate(row2, 1): wsP.cell(2, j, v)
for j, v in enumerate(["Column1", "Column2"] + [f"Sum of {c}" for c in pivot_value_cols], 1):
    wsP.cell(3, j, v)
for r, (_, pr) in enumerate(pivot.iterrows(), 4):
    wsP.cell(r, 1, pr.Column1); wsP.cell(r, 2, pr.Column2)
    for j, c in enumerate(pivot_value_cols, 3): wsP.cell(r, j, int(pr[c]))

wsD = wb.create_sheet("Data")
for j, name in enumerate(col_names, 1): wsD.cell(1, j, name)
for j, v in enumerate(category_row, 1): wsD.cell(2, j, v)   # blanks kept as "" (matches reference)
for j, v in enumerate(month_row, 1):    wsD.cell(3, j, v)
wsD.row_dimensions[2].hidden = True; wsD.row_dimensions[3].hidden = True

vi = {c: i for i, c in enumerate(col_names)}
for r in range(len(data_clean)):
    er = r + 4; v1 = data_clean.iloc[r].Column1; v2 = data_clean.iloc[r].Column2
    for c in col_names:
        j = vi[c] + 1
        if c in label_cols:
            wsD.cell(er, j, data_clean.iloc[r][c])
        else:
            num = data_num.iloc[r][c]; wsD.cell(er, j, None if pd.isna(num) else int(num))
    if v1 in COL1_DESELECT or v2 in COL2_DESELECT or str(v1).strip() == "" or str(v2).strip() == "":
        wsD.row_dimensions[er].hidden = True

last_row = 3 + len(data_clean); lc = get_column_letter(len(col_names))
wsD.auto_filter.ref = f"A1:{lc}{last_row}"
wsD.auto_filter.add_filter_column(0, sorted(set(data_clean.Column1) - COL1_DESELECT - {""}), blank=False)
wsD.auto_filter.add_filter_column(1, sorted(set(data_clean.Column2) - COL2_DESELECT - {""}), blank=False)
for ws in (wsP, wsD):
    for row in ws.iter_rows():
        for cell in row:
            if cell.value is not None: cell.font = Font(name="Arial", size=10)
wb.save(EXTRACT_XLSX)

dataextract_df = pd.read_excel(EXTRACT_XLSX, sheet_name="Data", header=0)
print("Saved:", EXTRACT_XLSX, "| dataextract_df:", dataextract_df.shape)

Saved: /mnt/user-data/outputs/Switzerland_Data_Extract_Automated.xlsx | dataextract_df: (10628, 74)


## Step — `SwitzerlandMapping_df`
Read the mapping (dimension) file. Header is on the 2nd row. Provides product→attribute and brick→geography lookups.

In [8]:
SwitzerlandMapping_df = pd.read_excel(MAPPING_XLSX, sheet_name="Switzerland", header=1)
print("SwitzerlandMapping_df:", SwitzerlandMapping_df.shape)
[c for c in SwitzerlandMapping_df.columns if not str(c).startswith("Unnamed")]

SwitzerlandMapping_df: (50, 39)


['Product / Class',
 'Brand',
 'Corporation',
 'Company',
 'Pack original',
 'International pack',
 'Local pack',
 'HU conversion factor',
 'ATC Class',
 'Category',
 'Sub-market',
 'Formulation original',
 'Formulation',
 'Molecule',
 'Ingredient',
 'Parallel import',
 'Reps',
 'RSM',
 'GM/CM/NSM',
 'Brick',
 'Region',
 'Territory',
 'Need clarification on Pack']

### Build lookups from the mapping
- **Product → **: Brand, Company, Pack (International pack), Molecule, Category, Sub-Market, Formulation, Import (Parallel import).
- **Brick → **: Region, Territory.
- **Territory → RSM** (per your instruction, RSM comes from the mapping, not the external `[3]` file).

Anything not found resolves to `"-"`, replicating the original `IFERROR(...,"-")`.

In [9]:
mp = SwitzerlandMapping_df
prod_lu, brick_lu, terr_rsm = {}, {}, {}
for _, r in mp.iterrows():
    p = r.get("Product / Class")
    if isinstance(p, str) and p.strip():
        prod_lu[p] = {"Brand": r.get("Brand"), "Company": r.get("Company"),
                      "Pack": r.get("International pack"), "Molecule": r.get("Molecule"),
                      "Category": r.get("Category"), "Sub-Market": r.get("Sub-market"),
                      "Formulation": r.get("Formulation"), "Import": r.get("Parallel import")}
    b = r.get("Brick")
    if isinstance(b, str) and b.strip():
        brick_lu[b] = {"Region": r.get("Region"), "Territory": r.get("Territory")}
    t = r.get("Territory")
    if isinstance(t, str) and t.strip():
        terr_rsm.setdefault(t, r.get("RSM"))

def g(d, k, f="-"):
    v = d.get(k) if d else None
    return f if v is None or (isinstance(v, float) and pd.isna(v)) else v

## Step 5 — Linked Data sheet (Option B: resolved to values)
Each Pivot row (Brick × Product, with its 24 Units + 24 Revenue months) is enriched with the mapping attributes and written as **actual values** — no live external formulas. Column order and names match the original Linked file's Data sheet (first 16 dimension columns), with month value columns labelled `UN-<Mon YYYY>` / `REV-<Mon YYYY>`.

In [10]:
rows = []
for _, pr in pivot.iterrows():
    brick, prod = pr.Column1, pr.Column2
    pl, bl = prod_lu.get(prod), brick_lu.get(brick)
    cat = g(pl, "Category")
    rec = {
        "Retina market": "Inculde" if isinstance(cat, str) and "Retina" in cat else "Exclude",
        "Region":     g(bl, "Region"),
        "RSM":        terr_rsm.get(g(bl, "Territory"), "-") if bl else "-",
        "Territory":  g(bl, "Territory"),
        "Brick":      brick,
        "ATC4 Class": "-",
        "Category":   cat,
        "Product":    prod,
        "Brand":      g(pl, "Brand"),
        "Company":    g(pl, "Company"),
        "Pack":       g(pl, "Pack"),
        "Molecule":   g(pl, "Molecule"),
        "Sub-Market": g(pl, "Sub-Market"),
        "Formulation":g(pl, "Formulation"),
        "Ingredient": "-",
        "Import":     g(pl, "Import"),
    }
    for lbl, c in zip(win_labels, units_src_cols): rec[f"UN-{lbl}"]  = int(pr[c])
    for lbl, c in zip(win_labels, rev_src_cols):   rec[f"REV-{lbl}"] = int(pr[c])
    rows.append(rec)

linked_cols = ["Retina market","Region","RSM","Territory","Brick","ATC4 Class","Category","Product",
               "Brand","Company","Pack","Molecule","Sub-Market","Formulation","Ingredient","Import"] \
              + [f"UN-{l}" for l in win_labels] + [f"REV-{l}" for l in win_labels]
CountryDashboardData_Switzerland_Linked_df = pd.DataFrame(rows, columns=linked_cols)

with pd.ExcelWriter(LINKED_XLSX, engine="openpyxl") as xw:
    CountryDashboardData_Switzerland_Linked_df.to_excel(xw, sheet_name="Data", index=False)

print("Saved:", LINKED_XLSX)
print("CountryDashboardData_Switzerland_Linked_df:", CountryDashboardData_Switzerland_Linked_df.shape)
CountryDashboardData_Switzerland_Linked_df.head()

Saved: /mnt/user-data/outputs/Country_Dashboard_Data_-_Switzerland_Linked_Automated.xlsx
CountryDashboardData_Switzerland_Linked_df: (9534, 64)


,Retina market,Region,RSM,Territory,Brick,ATC4 Class,Category,Product,Brand,Company,...,REV-Sep 2025,REV-Oct 2025,REV-Nov 2025,REV-Dec 2025,REV-Jan 2026,REV-Feb 2026,REV-Mar 2026,REV-Apr 2026,REV-May 2026,REV-Jun 2026
0,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,-,Glaucoma,All others S01E2 Myotica & Mittel zur Glaukomb...,OTHER GLAUCOMA,OTHERS,...,-102,-94,-65,90,-85,25,166,-130,-9,271
1,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,-,Glaucoma,Alphagan,ALPHAGAN,ALLERGAN,...,366,454,371,454,287,206,209,281,258,238
2,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,-,Glaucoma,Arteoptic,ARTEOPTIC,BAUSCH HEALTH,...,173,273,95,79,99,200,187,126,61,75
3,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,-,Glaucoma,Azarga,AZARGA,ALCON,...,363,189,489,370,163,367,184,216,316,528
4,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,-,Glaucoma,Azopt,AZOPT,NOVARTIS,...,497,395,421,360,195,301,538,491,292,492


## Validation
- Linked value totals equal the Pivot totals (nothing lost in the join).
- Brick/product coverage from the mapping (flags the `-` fallbacks).

In [11]:
u_tot_p = int(pivot[units_src_cols].sum().sum()); r_tot_p = int(pivot[rev_src_cols].sum().sum())
u_tot_l = int(CountryDashboardData_Switzerland_Linked_df[[f"UN-{l}"  for l in win_labels]].sum().sum())
r_tot_l = int(CountryDashboardData_Switzerland_Linked_df[[f"REV-{l}" for l in win_labels]].sum().sum())
print(f"Units   total  pivot={u_tot_p:,}  linked={u_tot_l:,}  match={u_tot_p==u_tot_l}")
print(f"Revenue total  pivot={r_tot_p:,}  linked={r_tot_l:,}  match={r_tot_p==r_tot_l}")

ld = CountryDashboardData_Switzerland_Linked_df
print(f"rows={len(ld)}  | Brand resolved={ (ld.Brand!='-').sum() }/{len(ld)}"
      f"  | Region resolved={ (ld.Region!='-').sum() }/{len(ld)}")
n_bricks = ld.Brick.nunique(); mapped_bricks = ld.loc[ld.Region!='-','Brick'].nunique()
print(f"bricks: {mapped_bricks}/{n_bricks} present in mapping geography "
      f"(remaining {n_bricks-mapped_bricks} resolve Region/Territory/RSM to '-')")

Units   total  pivot=2,250,285  linked=2,250,285  match=True
Revenue total  pivot=62,413,401  linked=62,413,401  match=True
rows=9534  | Brand resolved=9534/9534  | Region resolved=1722/9534
bricks: 41/227 present in mapping geography (remaining 186 resolve Region/Territory/RSM to '-')


### Notes / flagged edge cases
- **Brick coverage:** the provided Mapping file's geography block contains **41** bricks; the Pivot has **227**. The 186 unmapped bricks resolve Region/Territory/RSM to `"-"` (faithful to the original `IFERROR`). If they should populate, the Mapping's geography block needs the full brick list.
- **Territory/RSM:** in the provided mapping Territory is uniformly `Jurica Tripalo` and RSM is `"-"`, so RSM is `"-"` throughout.
- **`Retina market`:** reproduces the original formula's output (`"Exclude"` / `"Inculde"` — the source keeps that spelling); all current rows are `"Exclude"` since no category contains "Retina".
- **Row order** is Brick→Product (cosmetic; totals are order-independent).
- **Data1** and the `[1]` time-periods / `[5]` brands sources are intentionally out of scope for this step.

## Step 6 — Final dashboard (`Country_Dashboard_Switzerland_df`)

Build the final `Country Dashboard - Switzerland` file (36 months):

- **Dimensions + latest 24 months** come from the Linked df (value-paste).
- **Older 12 months** (frozen history) are pulled from the **existing dashboard**. They *cannot* be recomputed from the current raw file — the raw's older months are restated, so recomputing them disagrees with the published history. They must be carried forward from the prior dashboard.

**How rows are matched to the existing dashboard:** the existing file has no `Product` column, so we can't join on it directly, and ~11% of rows disagree on mapping-derived attributes. But every brick in the existing file lists the same 42 products in a **fixed slot order**, and the two files carry identical values for the months they share. We fingerprint each row on (brick + the shared months) to learn, by majority vote, which product sits in each slot, then pull each row's 12-month history by (brick, slot). Keying on the *shared* months (all 24 this cycle, 23 next cycle, …) keeps the match working every month with no code changes.

In [12]:
from collections import Counter, defaultdict

ex = pd.read_excel(EXISTING_DASHBOARD, sheet_name=0, header=0)

# month label lists
all_labels   = [en_month(month_labels[i]) for i in range(n_months)]
final_labels = all_labels[end_pos - TOTAL_MONTHS + 1 : end_pos + 1]   # 36 months
hist_labels  = final_labels[:HISTORY_MONTHS]                           # older 12 (from existing)
win_labels_36 = final_labels[HISTORY_MONTHS:]                         # latest 24 (from Linked)
assert win_labels_36 == win_labels, "window mismatch"

# --- Fingerprint on the months the two files SHARE ---------------------------
# The existing dashboard and the Linked df carry identical values for the months
# they have in common. This month they overlap on all 24; next month only 23,
# etc. Keying on the shared months keeps the match working every cycle untouched.
ex_month_cols = [c for c in ex.columns if isinstance(c, str) and c.startswith("UN-")]
ex_months     = [c[3:] for c in ex_month_cols]                       # labels present in existing file
shared_months = [l for l in win_labels if l in ex_months]            # overlap with Linked window
assert len(shared_months) >= 3, "too few overlapping months to fingerprint rows reliably"
print(f"months shared with existing dashboard: {len(shared_months)} "
      f"({shared_months[0]} -> {shared_months[-1]})")

vcols = [f"UN-{l}" for l in shared_months] + [f"REV-{l}" for l in shared_months]
def vkey(df):
    return df["Brick"].astype(str) + "#" + df[vcols].fillna(0).astype(float).astype(int).astype(str).agg(",".join, axis=1)
ex = ex.copy(); ex["__vk"] = vkey(ex)
ex["__pos"] = ex.groupby("Brick", sort=False).cumcount()
lkv = CountryDashboardData_Switzerland_Linked_df.copy(); lkv["__vk"] = vkey(lkv)

# (brick, shared-months vector) -> product(s) from the Linked df
vk_to_products = defaultdict(list)
for _, r in lkv.iterrows():
    vk_to_products[r["__vk"]].append(r["Product"])

# slot -> product by majority vote across bricks (only unambiguous matches vote)
slot_votes = defaultdict(Counter)
for _, r in ex.iterrows():
    cands = vk_to_products.get(r["__vk"], [])
    if len(cands) == 1:
        slot_votes[r["__pos"]][cands[0]] += 1
slot_to_product = {p: slot_votes[p].most_common(1)[0][0] for p in range(len(slot_votes))}
product_to_slot = {v: k for k, v in slot_to_product.items()}
assert len(set(slot_to_product.values())) == pivot.Column2.nunique(), "slot map not bijective"
print(f"slot->product map: {len(slot_to_product)} slots, all products distinct")

# existing-file history keyed by (brick, slot)
ex_hist = {(r["Brick"], r["__pos"]): r for _, r in ex.iterrows()}

months shared with existing dashboard: 24 (Jul 2024 -> Jun 2026)


slot->product map: 42 slots, all products distinct


### Assemble the final rows + derived period block
`Key = Brand & Formulation`, `Key2 = Territory & Brick` (values). Dimensions and the latest 24 months come from the Linked df; the older 12 months from the existing dashboard.

The **period block** (cols 88–129) is computed for three anchors — Current (latest month), Previous Period (−1 month), Previous Year (−12 months) — each with 7 measures, for UN and REV. Fiscal year starts **April**, so quarters are Apr–Jun / Jul–Sep / Oct–Dec / Jan–Mar and terciles are Apr–Jul / Aug–Nov / Dec–Mar.
- **Plain** = the month's value.
- **Fixed Qtr / Fixed Tercile** = sum from the first month of the current fiscal quarter/tercile through the anchor.
- **Rolling Qtr** = trailing 3 months; **Rolling Tercile** = trailing 4 months.
- **FY YTD** = sum from the most recent April through the anchor.
- **MAT** = trailing 12 months.

The trailing calendar-helper block (cols 131–159) is carried over from the existing dashboard as-is.

In [13]:
# ---- fiscal-period helpers -------------------------------------------------
_M = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
_MI = {v: k for k, v in _M.items()}
def _o(label):                       # "Jun 2026" -> month ordinal
    mm, yy = label.split(); return int(yy) * 12 + (_M[mm] - 1)
def _fq_start(o):                    # first month of fiscal quarter (=calendar quarter)
    y, m = o // 12, o % 12 + 1;  return y * 12 + (((m - 1) // 3) * 3 + 1 - 1)
def _ft_start(o):                    # first month of fiscal tercile (Apr-Jul/Aug-Nov/Dec-Mar)
    y, m = o // 12, o % 12 + 1
    s = (y, 4) if 4 <= m <= 7 else (y, 8) if 8 <= m <= 11 else (y, 12) if m == 12 else (y - 1, 12)
    return s[0] * 12 + (s[1] - 1)
def _fy_start(o):                    # most recent April (fiscal-year start)
    y, m = o // 12, o % 12 + 1;  s = (y, 4) if m >= 4 else (y - 1, 4);  return s[0] * 12 + (s[1] - 1)
def _rsum(d, a, b): return sum(d.get(o, 0) for o in range(a, b + 1))
def _measure(d, o, kind):
    if kind == "plain": return d.get(o, 0)
    if kind == "fq":    return _rsum(d, _fq_start(o), o)      # Fixed Qtr
    if kind == "rq":    return _rsum(d, o - 2, o)             # Rolling Qtr (3 mo)
    if kind == "ft":    return _rsum(d, _ft_start(o), o)      # Fixed Tercile
    if kind == "rt":    return _rsum(d, o - 3, o)             # Rolling Tercile (4 mo)
    if kind == "ytd":   return _rsum(d, _fy_start(o), o)      # FY YTD
    if kind == "mat":   return _rsum(d, o - 11, o)            # MAT (12 mo)

# anchors, measures, and exact column-label templates
cur_lbl = final_labels[-1]
pp_lbl  = _MI[(_o(cur_lbl) - 1) % 12 + 1]  + f" {(_o(cur_lbl) - 1) // 12}"
py_lbl  = _MI[(_o(cur_lbl) - 12) % 12 + 1] + f" {(_o(cur_lbl) - 12) // 12}"
_anchors = [("Current", cur_lbl), ("Previous Period", pp_lbl), ("Previous Year", py_lbl)]
_kinds   = [("plain",""),("fq","Fixed Qtr "),("rq","Rolling Qtr "),("ft","Fixed Tercile "),
            ("rt","Rolling Tercile "),("ytd","FY YTD "),("mat","MAT ")]
# period columns in the file's order: UN (3 anchors x 7), then REV (3 anchors x 7)
period_cols = [f"{an} {met}-{kl}{am}"
               for met in ["UN","REV"] for an, am in _anchors for _, kl in _kinds]

final_dims = ["Retina market","Region","RSM","Territory","Brick","Category","Brand","Company",
              "Pack","Molecule","Sub-Market","Formulation","Ingredient","Import"]

recs = []
for _, r in CountryDashboardData_Switzerland_Linked_df.iterrows():
    slot = product_to_slot[r["Product"]]
    jr = ex_hist.get((r["Brick"], slot))
    rec = {"Key": f"{r['Brand']}{r['Formulation']}"}
    for d in final_dims:
        rec[d] = r[d]
    # 36 months: older 12 from existing dashboard, latest 24 from Linked df
    un_series, rev_series = {}, {}
    for l in hist_labels:
        u = int(jr[f"UN-{l}"])  if jr is not None and pd.notna(jr[f"UN-{l}"])  else 0
        v = int(jr[f"REV-{l}"]) if jr is not None and pd.notna(jr[f"REV-{l}"]) else 0
        rec[f"UN-{l}"] = u; rec[f"REV-{l}"] = v; un_series[_o(l)] = u; rev_series[_o(l)] = v
    for l in win_labels:
        u = int(r[f"UN-{l}"]); v = int(r[f"REV-{l}"])
        rec[f"UN-{l}"] = u; rec[f"REV-{l}"] = v; un_series[_o(l)] = u; rev_series[_o(l)] = v
    # derived period block
    for met, series in [("UN", un_series), ("REV", rev_series)]:
        for an, am in _anchors:
            o = _o(am)
            for k, kl in _kinds:
                rec[f"{an} {met}-{kl}{am}"] = _measure(series, o, k)
    rec["Key2"] = f"{r['Territory']}{r['Brick']}"
    recs.append(rec)

final_cols = (["Key"] + final_dims
              + [f"UN-{l}"  for l in final_labels]
              + [f"REV-{l}" for l in final_labels]
              + period_cols + ["Key2"])
Country_Dashboard_Switzerland_df = pd.DataFrame(recs, columns=final_cols)

# ---- write file, then carry over the trailing helper block (cols 131-159) ----
import openpyxl
with pd.ExcelWriter(FINAL_XLSX, engine="openpyxl") as xw:
    Country_Dashboard_Switzerland_df.to_excel(xw, sheet_name="Sheet1", index=False)

_src = openpyxl.load_workbook(EXISTING_DASHBOARD, data_only=True)["Sheet1"]
n_out = len(final_cols)                       # last written column (Key2 at col 130)
if _src.max_column > n_out:
    _wb = openpyxl.load_workbook(FINAL_XLSX); _ws = _wb["Sheet1"]
    for col in range(n_out + 1, _src.max_column + 1):
        for row in range(1, _src.max_row + 1):
            v = _src.cell(row, col).value
            if v is not None:
                _ws.cell(row, col, v)
    _wb.save(FINAL_XLSX)

print("Saved:", FINAL_XLSX)
print("Country_Dashboard_Switzerland_df:", Country_Dashboard_Switzerland_df.shape,
      f"| {TOTAL_MONTHS} months {final_labels[0]}->{final_labels[-1]} | period anchors:",
      cur_lbl, "/", pp_lbl, "/", py_lbl)
Country_Dashboard_Switzerland_df.head()

Saved: /mnt/user-data/outputs/Country_Dashboard_-_Switzerland_FINAL.xlsx
Country_Dashboard_Switzerland_df: (9534, 130) | 36 months Jul 2023->Jun 2026 | period anchors: Jun 2026 / May 2026 / Jun 2025


,Key,Retina market,Region,RSM,Territory,Brick,Category,Brand,Company,Pack,...,Previous Period REV-FY YTD May 2026,Previous Period REV-MAT May 2026,Previous Year REV-Jun 2025,Previous Year REV-Fixed Qtr Jun 2025,Previous Year REV-Rolling Qtr Jun 2025,Previous Year REV-Fixed Tercile Jun 2025,Previous Year REV-Rolling Tercile Jun 2025,Previous Year REV-FY YTD Jun 2025,Previous Year REV-MAT Jun 2025,Key2
0,OTHER GLAUCOMA-,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,Glaucoma,OTHER GLAUCOMA,OTHERS,-,...,-139,2,148,-111,-111,-111,-168,-111,-106,Jurica Tripalo2001 GE Genève 1201
1,ALPHAGANMD,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,Glaucoma,ALPHAGAN,ALLERGAN,OPH DRP 0.2% 1 5ML,...,539,3757,179,741,741,741,922,741,2610,Jurica Tripalo2001 GE Genève 1201
2,ARTEOPTICMD,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,Glaucoma,ARTEOPTIC,BAUSCH HEALTH,OPH DRP 2% 3 5ML,...,187,1728,96,616,616,616,782,616,2271,Jurica Tripalo2001 GE Genève 1201
3,AZARGAMD,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,Glaucoma,AZARGA,ALCON,OPH SUSP 1%+0.5% 1 5ML,...,532,3482,138,763,763,763,1238,763,4164,Jurica Tripalo2001 GE Genève 1201
4,AZOPTMD,Exclude,Switzerland,-,Jurica Tripalo,2001 GE Genève 1201,Glaucoma,AZOPT,NOVARTIS,OPH SUSP 1% 1 5ML,...,783,4645,512,1289,1289,1289,1594,1289,5000,Jurica Tripalo2001 GE Genève 1201


### Validation vs the existing dashboard
For rows whose mapping-derived dimensions agree with the existing file, every one of the 36 months **and** all 42 period columns should match — except one known reference-file quirk (see note).

In [14]:
dims_chk = ["Brick","Category","Brand","Company","Pack","Molecule","Sub-Market","Formulation","Import"]
a = Country_Dashboard_Switzerland_df.copy(); a["sig"] = a[dims_chk].astype(str).agg("||".join, axis=1)
b = ex.copy();                              b["sig"] = b[dims_chk].astype(str).agg("||".join, axis=1)
m = a.merge(b, on="sig", suffixes=("_new", "_ex"))

month_cols  = [f"UN-{l}" for l in final_labels] + [f"REV-{l}" for l in final_labels]
from collections import Counter
mism_by_col = Counter()
for c in month_cols + period_cols:
    if c + "_ex" in m.columns:
        x = pd.to_numeric(m[c+"_new"], errors="coerce").fillna(0)
        y = pd.to_numeric(m[c+"_ex"],  errors="coerce").fillna(0)
        n = int((x - y).abs().gt(0).sum())
        if n: mism_by_col[c] = n

print(f"dimension-matched rows: {len(m)} / {len(a)}")
print(f"36-month value mismatches: {sum(mism_by_col[c] for c in month_cols)}")
print(f"period-block mismatches: {sum(mism_by_col[c] for c in period_cols)}")
if mism_by_col:
    print("columns with mismatches:")
    for c, n in mism_by_col.items(): print(f"  {c}: {n} rows")
print(f"rows with accepted mapping-derived dimension differences: {len(a) - len(m)}")

dimension-matched rows: 8466 / 9534
36-month value mismatches: 0
period-block mismatches: 6088
columns with mismatches:
  Previous Period REV-Rolling Tercile May 2026: 6088 rows
rows with accepted mapping-derived dimension differences: 1068


### Notes
- **History source:** the older 12 months are frozen from the existing dashboard, *not* recomputed from raw (recomputing disagreed by ~138k cells due to restatements).
- **`EXISTING_DASHBOARD`** should point at the previous month's published dashboard each cycle. Row matching uses the shared-months fingerprint + fixed slot order, so it carries history forward without a `Product` column and needs no edits month to month.
- **Reference-file quirk:** in the June file the single column `Previous Period REV-Rolling Tercile` sums only 3 months; every other rolling-tercile column (and your definition) uses 4. This notebook follows the spec (4 months) everywhere, so that one column intentionally differs from the June file — all other 41 period columns match exactly.
- **Deferred → now done:** the derived period block (cols 88–129) is computed here. The trailing calendar-helper block (cols 131–159) is carried over from the existing dashboard verbatim.
- The ~11% dimension differences are the accepted mapping-coverage issue, not a pipeline error; value and period columns match wherever dimensions agree.